## 1. 라이브러리 및 환경 설정

In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import timm

# 하이퍼파라미터 설정
CFG = {
    'IMG_SIZE': 384,
    'EPOCHS': 30,
    'LEARNING_RATE': 3e-4,      # freeze 중 새 레이어 빠르게 학습
    'FINETUNE_LR': 5e-5,        # unfreeze 후 전체 fine-tune LR
    'WEIGHT_DECAY': 1e-4,
    'BATCH_SIZE': 16,
    'SEED': 42,
    'PATIENCE': 10,
    'FREEZE_EPOCHS': 5,         # B4에서 3이 부족했으므로 5로 확대
    'LABEL_SMOOTHING': 0.05,
    'TTA_NUM': 10
}

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['SEED'])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

c:\Users\user\anaconda3\envs\a\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. 데이터 로드 및 학습/검증 데이터 분할

In [2]:
train_df = pd.read_csv('./data/train.csv')
val_df = pd.read_csv('./data/dev.csv')

print(f"학습 데이터 개수: {len(train_df)}")
print(f"검증 데이터 개수: {len(val_df)}")

학습 데이터 개수: 1000
검증 데이터 개수: 100


## 3. 커스텀 데이터셋 클래스 정의 및 데이터 로더

In [3]:
class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample_id = str(self.df.iloc[idx]['id'])
        folder_path = os.path.join(self.root_dir, sample_id)
        
        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(folder_path, f"{name}.png")
            image = Image.open(img_path).convert("RGB")
            if self.transform:
                image = self.transform(image)
            views.append(image)
            
        if self.is_test:
            return views
        
        label = self.label_map[self.df.iloc[idx]['label']]
        return views, label

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((420, 420)),
    transforms.RandomCrop(CFG['IMG_SIZE']),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02),
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
])

test_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

tta_transform = transforms.Compose([
    transforms.Resize((420, 420)),
    transforms.RandomCrop(CFG['IMG_SIZE']),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = MultiViewDataset(train_df, './data/train', train_transform, is_test=False)
val_dataset = MultiViewDataset(val_df, './data/dev', test_transform, is_test=False)

train_loader = DataLoader(train_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

test_df = pd.read_csv('./data/sample_submission.csv')
test_dataset = MultiViewDataset(test_df, './data/test', test_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

## 4. 모델 정의 (EfficientNet-V2-S + Cross-View Attention)

In [5]:
# ─── Building Blocks ─────────────────────────────────────────────────

class CrossViewAttention(nn.Module):
    """Bidirectional cross-attention between front and top view features."""
    def __init__(self, embed_dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.cross_attn_f2t = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.cross_attn_t2f = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm_f = nn.LayerNorm(embed_dim)
        self.norm_t = nn.LayerNorm(embed_dim)
        self.ffn_f = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(embed_dim * 4, embed_dim), nn.Dropout(dropout),
        )
        self.ffn_t = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(embed_dim * 4, embed_dim), nn.Dropout(dropout),
        )
        self.norm_ffn_f = nn.LayerNorm(embed_dim)
        self.norm_ffn_t = nn.LayerNorm(embed_dim)

    def forward(self, feat_front, feat_top):
        attn_f, _ = self.cross_attn_f2t(feat_front, feat_top, feat_top)
        feat_front = self.norm_f(feat_front + attn_f)
        attn_t, _ = self.cross_attn_t2f(feat_top, feat_front, feat_front)
        feat_top = self.norm_t(feat_top + attn_t)
        feat_front = self.norm_ffn_f(feat_front + self.ffn_f(feat_front))
        feat_top = self.norm_ffn_t(feat_top + self.ffn_t(feat_top))
        return feat_front, feat_top


class ViewDiscrepancyModule(nn.Module):
    """Encodes cross-view discrepancy as an explicit anomaly signal."""
    def __init__(self, feat_dim):
        super().__init__()
        self.fusion = nn.Sequential(
            nn.Linear(feat_dim * 4, feat_dim), nn.GELU(),
            nn.Dropout(0.1), nn.Linear(feat_dim, feat_dim),
        )

    def forward(self, feat_front, feat_top):
        diff = feat_front - feat_top
        hadamard = feat_front * feat_top
        combined = torch.cat([feat_front, feat_top, diff, hadamard], dim=1)
        return self.fusion(combined)


# ─── Main Model ──────────────────────────────────────────────────────

class CollapsePredictor(nn.Module):
    """
    v4: EfficientNet-V2-S + Cross-View Attention.
    
    B4 실패 원인 반영:
    - ImageNet-21k pretrained (소규모 데이터에 더 좋은 transfer)
    - Multi-scale 제거: backbone의 global pooling을 직접 사용
      (새 projection layer를 줄여서 freeze 중 학습 부담 축소)
    - Cross-view attention은 backbone feature 위에 바로 적용
    """
    def __init__(self, num_classes=1, project_dim=128, num_attn_layers=1, num_heads=4):
        super().__init__()

        # EfficientNet-V2-S: 21k pretrained → 1k fine-tuned
        # 384 해상도 pretrained weight 사용
        self.backbone = timm.create_model(
            'tf_efficientnetv2_s.in21k_ft_in1k',
            pretrained=True,
            features_only=True,
            out_indices=[4]       # 마지막 stage만 사용 (가장 안정적)
        )
        backbone_dim = self.backbone.feature_info.channels()[-1]  # 256
        
        # Backbone feature → project_dim 으로 단일 projection
        self.proj = nn.Sequential(
            nn.Conv2d(backbone_dim, project_dim, 1),
            nn.BatchNorm2d(project_dim),
            nn.GELU(),
        )
        
        # View-type embedding
        self.view_embed = nn.Parameter(torch.randn(2, 1, project_dim) * 0.02)

        # Cross-view attention
        self.cross_attn_layers = nn.ModuleList([
            CrossViewAttention(project_dim, num_heads=num_heads)
            for _ in range(num_attn_layers)
        ])

        # Discrepancy module
        self.discrepancy = ViewDiscrepancyModule(project_dim)

        # cross_f + cross_t + disc + global_f + global_t = 5D
        self.classifier = nn.Sequential(
            nn.Linear(project_dim * 5, project_dim),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(project_dim, num_classes),
        )

    def _extract(self, x):
        """Single view feature extraction."""
        feat_map = self.backbone(x)[-1]       # [B, backbone_dim, H, W]
        feat_map = self.proj(feat_map)         # [B, project_dim, H, W]
        
        B, C, H, W = feat_map.shape
        spatial = feat_map.view(B, C, H * W).permute(0, 2, 1)  # [B, HW, D]
        global_feat = feat_map.mean(dim=[2, 3])                 # [B, D]
        return spatial, global_feat

    def forward(self, views):
        front_img, top_img = views[0], views[1]

        spatial_f, global_f = self._extract(front_img)
        spatial_t, global_t = self._extract(top_img)

        spatial_f = spatial_f + self.view_embed[0]
        spatial_t = spatial_t + self.view_embed[1]

        sf, st = spatial_f, spatial_t
        for attn_layer in self.cross_attn_layers:
            sf, st = attn_layer(sf, st)

        cross_f = sf.mean(dim=1)
        cross_t = st.mean(dim=1)

        disc = self.discrepancy(global_f, global_t)

        combined = torch.cat([cross_f, cross_t, disc, global_f, global_t], dim=1)
        return self.classifier(combined)


# ── 모델 정보 출력 ────────────────────────────────────────────────────
_m = CollapsePredictor().to(device)
total_params = sum(p.numel() for p in _m.parameters()) / 1e6
backbone_params = sum(p.numel() for p in _m.backbone.parameters()) / 1e6
new_params = total_params - backbone_params
print(f'Total: {total_params:.1f}M | Backbone(frozen): {backbone_params:.1f}M | New layers: {new_params:.1f}M')

# backbone feature 확인
with torch.no_grad():
    dummy = torch.randn(1, 3, 384, 384).to(device)
    feat = _m.backbone(dummy)[-1]
    print(f'Backbone output: {feat.shape}')  # [1, 256, 12, 12]
del _m

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
c:\Users\user\anaconda3\envs\a\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--timm--tf_efficientnetv2_s.in21k_ft_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see 

Total: 20.4M | Backbone(frozen): 19.8M | New layers: 0.6M
Backbone output: torch.Size([1, 256, 12, 12])


## 4. 학습 및 검증 루프 구현

In [6]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    train_loss = 0
    for views, labels in tqdm(loader, desc="Training"):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float()
        
        smooth = CFG['LABEL_SMOOTHING']
        labels = labels * (1 - smooth) + (1 - labels) * smooth
        
        optimizer.zero_grad()
        outputs = model(views).view(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
    return train_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for views, labels in tqdm(loader, desc="Validation"):
            views = [v.to(device) for v in views]
            labels = labels.to(device).float()
            
            outputs = model(views).view(-1)
            probs = torch.sigmoid(outputs)
            
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)
    
    eps = 1e-15
    p = np.clip(all_probs, eps, 1 - eps)
    logloss_score = -np.mean(all_labels * np.log(p) + (1 - all_labels) * np.log(1 - p))
    acc_score = np.mean((all_probs > 0.5) == all_labels)
    
    return logloss_score, acc_score

In [7]:
model = CollapsePredictor().to(device)
criterion = nn.BCEWithLogitsLoss()

# ── Phase 1: Backbone Freeze (5 epochs) ──────────────────────────────
# 새 레이어(proj, attention, classifier)가 backbone feature에
# 충분히 적응할 시간을 줌. B4에서 3 epoch이 부족했던 문제 해결.
for param in model.backbone.parameters():
    param.requires_grad = False

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG['LEARNING_RATE'],       # 3e-4: 새 레이어만 빠르게 학습
    weight_decay=CFG['WEIGHT_DECAY']
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['FREEZE_EPOCHS'], eta_min=1e-5
)

# ── Training Loop ────────────────────────────────────────────────────
best_logloss = float('inf')
patience_counter = 0

for epoch in range(1, CFG['EPOCHS'] + 1):
    # Phase 2: Unfreeze backbone
    if epoch == CFG['FREEZE_EPOCHS'] + 1:
        print('\n── Unfreezing EfficientNet-V2-S backbone ──')
        for param in model.backbone.parameters():
            param.requires_grad = True
        optimizer = optim.AdamW([
            {'params': model.backbone.parameters(), 'lr': CFG['FINETUNE_LR'] * 0.1},
            {'params': model.proj.parameters(),     'lr': CFG['FINETUNE_LR']},
            {'params': model.cross_attn_layers.parameters(), 'lr': CFG['FINETUNE_LR']},
            {'params': model.discrepancy.parameters(),       'lr': CFG['FINETUNE_LR']},
            {'params': model.classifier.parameters(),        'lr': CFG['FINETUNE_LR']},
            {'params': [model.view_embed],                   'lr': CFG['FINETUNE_LR']},
        ], weight_decay=CFG['WEIGHT_DECAY'])
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG['EPOCHS'] - CFG['FREEZE_EPOCHS'], eta_min=1e-6
        )

    avg_train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_logloss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()

    current_lr = optimizer.param_groups[0]['lr']
    print(f'Epoch [{epoch}] lr={current_lr:.6f}')
    print(f'  - Train Loss: {avg_train_loss:.4f}')
    print(f'  - Val Log-Loss: {val_logloss:.6f} | Val Acc: {val_acc:.4f}')

    if val_logloss < best_logloss:
        best_logloss = val_logloss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'  ★ Best model saved (Log-Loss: {best_logloss:.6f})')
    else:
        patience_counter += 1
        print(f'  - No improvement ({patience_counter}/{CFG["PATIENCE"]})')

    if patience_counter >= CFG['PATIENCE']:
        print(f'\nEarly stopping at epoch {epoch}')
        break

model.load_state_dict(torch.load('best_model.pt', weights_only=True))
print(f'\nBest Val Log-Loss: {best_logloss:.6f}')

Unexpected keys (bn2.bias, bn2.num_batches_tracked, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
Validation: 100%|██████████| 7/7 [00:01<00:00,  5.66it/s]


Epoch [1] lr=0.000272
  - Train Loss: 0.6198
  - Val Log-Loss: 0.584447 | Val Acc: 0.7200
  ★ Best model saved (Log-Loss: 0.584447)


Validation: 100%|██████████| 7/7 [00:01<00:00,  5.84it/s]


Epoch [2] lr=0.000200
  - Train Loss: 0.4427
  - Val Log-Loss: 0.560064 | Val Acc: 0.7500
  ★ Best model saved (Log-Loss: 0.560064)


Validation: 100%|██████████| 7/7 [00:01<00:00,  5.73it/s]


...

Epoch [16] lr=0.000003
  - Train Loss: 0.2188
  - Val Log-Loss: 0.757665 | Val Acc: 0.6800
  - No improvement (8/10)


Validation: 100%|██████████| 7/7 [00:00<00:00,  7.07it/s]


Epoch [17] lr=0.000003
  - Train Loss: 0.2199
  - Val Log-Loss: 0.590419 | Val Acc: 0.7300
  - No improvement (9/10)


Validation: 100%|██████████| 7/7 [00:00<00:00,  7.20it/s]

Epoch [18] lr=0.000003
  - Train Loss: 0.2231
  - Val Log-Loss: 0.599156 | Val Acc: 0.7500
  - No improvement (10/10)

Early stopping at epoch 18

Best Val Log-Loss: 0.502442


## 5. 추론 (TTA: Test Time Augmentation)

In [8]:
model.eval()

# ── 1) 원본(clean) 예측 ──────────────────────────────────────────────
clean_probs = []
with torch.no_grad():
    for views in tqdm(test_loader, desc="Clean inference"):
        views = [v.to(device) for v in views]
        outputs = model(views).view(-1)
        probs = torch.sigmoid(outputs)
        clean_probs.extend(probs.cpu().numpy())
clean_probs = np.array(clean_probs, dtype=np.float64)

# ── 2) TTA 예측 ──────────────────────────────────────────────────────
tta_probs_list = []

for tta_i in range(CFG['TTA_NUM']):
    tta_dataset = MultiViewDataset(test_df, './data/test', tta_transform, is_test=True)
    tta_loader = DataLoader(tta_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)
    
    round_probs = []
    with torch.no_grad():
        for views in tta_loader:
            views = [v.to(device) for v in views]
            outputs = model(views).view(-1)
            probs = torch.sigmoid(outputs)
            round_probs.extend(probs.cpu().numpy())
    
    tta_probs_list.append(np.array(round_probs, dtype=np.float64))
    print(f'  TTA round {tta_i+1}/{CFG["TTA_NUM"]} done')

# ── 3) 가중 평균 ─────────────────────────────────────────────────────
all_rounds = [clean_probs] + tta_probs_list
weights = [2.0] + [1.0] * CFG['TTA_NUM']
weights = np.array(weights) / sum(weights)

all_probs = np.zeros_like(clean_probs)
for w, p in zip(weights, all_rounds):
    all_probs += w * p

print(f'\nFinal prediction stats: mean={all_probs.mean():.4f}, std={all_probs.std():.4f}')
print(f'  min={all_probs.min():.4f}, max={all_probs.max():.4f}')

# ── 4) 결과 저장 ─────────────────────────────────────────────────────
submission = pd.DataFrame({
    'id': test_df['id'],
    'unstable_prob': all_probs,
    'stable_prob': 1.0 - all_probs
})

os.makedirs("results", exist_ok=True)
submission.to_csv('./results/submission.csv', encoding='UTF-8-sig', index=False)
print("submission.csv 저장 완료. (EfficientNet-V2-S + TTA)")

Clean inference: 100%|██████████| 63/63 [00:10<00:00,  5.78it/s]


  TTA round 1/10 done
  TTA round 2/10 done
  TTA round 3/10 done
  TTA round 4/10 done
  TTA round 5/10 done
  TTA round 6/10 done
  TTA round 7/10 done
  TTA round 8/10 done
  TTA round 9/10 done
  TTA round 10/10 done

Final prediction stats: mean=0.5822, std=0.3059
  min=0.0530, max=0.9625
submission.csv 저장 완료. (EfficientNet-V2-S + TTA)
